In [89]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,2283,1,14,1,10,1,14,1,2,1,...,6,10,13,10,11.800000,0,13.9,-0.3,0.79,Graduate
1,3874,1,12,1,8,1,1,1,1,1,...,0,5,5,0,0.000000,5,12.7,3.7,-1.70,Dropout
2,2281,1,15,1,4,1,14,1,1,28,...,0,6,15,4,10.500000,0,15.5,2.8,-4.06,Enrolled
3,817,1,7,1,5,1,1,1,1,1,...,0,6,6,6,12.833333,0,12.4,0.5,1.79,Graduate
4,404,1,8,1,12,1,1,1,22,27,...,0,8,8,7,14.731429,0,10.8,1.4,1.74,Graduate


In [90]:
df_train.columns

Index(['SampleID', 'Marital status', 'Application mode', 'Application order',
       'Course', 'Daytime/evening attendance', 'Previous qualification',
       'Nacionality', 'Mother's qualification', 'Father's qualification',
       'Mother's occupation', 'Father's occupation', 'Displaced',
       'Educational special needs', 'Debtor', 'Tuition fees up to date',
       'Gender', 'Scholarship holder', 'Age at enrollment', 'International',
       'Curricular units 1st sem (credited)',
       'Curricular units 1st sem (enrolled)',
       'Curricular units 1st sem (evaluations)',
       'Curricular units 1st sem (approved)',
       'Curricular units 1st sem (grade)',
       'Curricular units 1st sem (without evaluations)',
       'Curricular units 2nd sem (credited)',
       'Curricular units 2nd sem (enrolled)',
       'Curricular units 2nd sem (evaluations)',
       'Curricular units 2nd sem (approved)',
       'Curricular units 2nd sem (grade)',
       'Curricular units 2nd sem (without 

In [91]:
features = ['Marital status', 'Application mode', 'Application order','Course', 'Daytime/evening attendance', 'Previous qualification','Nacionality', "Mother's qualification", "Father's qualification",
       "Mother's occupation", "Father's occupation", 'Displaced','Educational special needs', 'Debtor', 'Tuition fees up to date','Gender', 'Scholarship holder', 'Age at enrollment', 'International',
       'Curricular units 1st sem (credited)','Curricular units 1st sem (enrolled)','Curricular units 1st sem (evaluations)','Curricular units 1st sem (approved)','Curricular units 1st sem (grade)',
       'Curricular units 1st sem (without evaluations)','Curricular units 2nd sem (credited)','Curricular units 2nd sem (enrolled)','Curricular units 2nd sem (evaluations)',
       'Curricular units 2nd sem (approved)','Curricular units 2nd sem (grade)','Curricular units 2nd sem (without evaluations)', 'Unemployment rate','Inflation rate', 'GDP']
target = 'Target'

X = df_train[features]
y = df_train[target].map({"Dropout": 0, "Enrolled": 1, "Graduate": 2})

X_final = df_test[features]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_final = pd.DataFrame(scaler.transform(X_final), columns=features)

from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP
532,-0.29202,0.963663,-0.556229,0.481661,0.347852,-0.390915,-0.142136,1.080819,0.954982,-0.833632,...,-0.210957,-0.283546,0.798941,-0.009543,0.847603,1.024952,-0.195530,-0.293287,0.128791,0.757833
3386,-0.29202,-1.110677,-0.556229,-0.209953,0.347852,-0.390915,-0.142136,0.081482,-0.223713,-0.329837,...,-0.210957,-0.283546,-0.552829,0.494538,-0.803833,0.342986,-0.195530,-1.496306,0.998625,0.132677
2163,-0.29202,-1.110677,2.504005,0.020585,0.347852,-0.390915,-0.142136,0.636669,-0.223713,-1.589324,...,-0.210957,-0.283546,-0.102239,-0.513624,0.517316,0.279310,-0.195530,-1.496306,0.998625,0.132677
1813,-0.29202,1.906545,-0.556229,-0.901568,0.347852,3.340071,-0.142136,0.081482,-1.402409,0.677753,...,-0.210957,1.796180,1.249531,1.250659,0.517316,0.534013,-0.195530,-1.007580,0.128791,1.537077
842,-0.29202,-0.544948,-0.556229,-2.054259,0.347852,3.091339,-0.142136,1.191857,0.954982,-0.329837,...,-0.210957,5.955631,4.403662,3.266981,2.499039,0.390743,1.208085,-0.180504,-0.451098,0.881104


In [92]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

model = CatBoostClassifier(random_state=42).fit(X, y)
y_preds = model.predict(X_valid)
f1_score(y_valid, y_preds, average='macro')

0.781926976598711

In [93]:
y_final = model.predict(X_final).ravel()

In [94]:
submission = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "Target": y_final
})

submission.Target = submission.Target.map({0: "Dropout", 1: "Enrolled", 2:"Graduate"})

submission

,SampleID,Target
0,1853,Graduate
1,2399,Graduate
2,510,Dropout
3,242,Graduate
4,3392,Graduate
...,...,...
880,555,Graduate
881,2418,Graduate
882,2367,Graduate
883,1801,Graduate


In [95]:
submission.to_csv("submission.csv", index=False)